In [1]:
    
import argparse, json


from model import fetch_model
from utils.flow_viz import flow_to_image
from utils.utils import load_ckpt, coords_grid, bilinear_sampler


cfg = 'config/a2/dinov3/chairs-things.json'
ckpt = 'weights/a2/waftv2-ckpts/dinov3/sintel.pth'

args = argparse.Namespace()
args_dict = args.__dict__

args_dict['ckpt'] = ckpt
args_dict['scale'] = 0.0

with open(cfg, 'r') as f:
    data = json.load(f)

    for key, value in data.items():
        args_dict[key] = value


model = fetch_model(args)
load_ckpt(model, args.ckpt)
model = model.cuda()
model.eval()
# wrapped_model = InferenceWrapper(model, scale=args.scale, train_size=args.image_size, pad_to_train_size=False, tiling=False)

xFormers not available
xFormers not available


WAFTv2(
  (encoder): DinoV3Feature(
    (encoder): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (rope_embed): RopePositionEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x SelfAttentionBlock(
          (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
          (attn): SelfAttention(
            (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=384, out_features=384, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=384, out_features=1536, bias=True)
            (act): GELU(approximate='none')
            (fc2): Linear(in_features=1536, out_

In [2]:
args.image_size

[432, 960]

In [3]:
from utils import frame_utils
import numpy as np
import torch, cv2

def preprocessing(image1_file, image2_file, is_test=True, size=(448,448)):
        if is_test:
            img1 = frame_utils.read_gen(image1_file)
            img2 = frame_utils.read_gen(image2_file)
            img1 = np.array(img1).astype(np.uint8)[..., :3]
            img2 = np.array(img2).astype(np.uint8)[..., :3]
            img1 = cv2.resize(img1, size) 
            img2 = cv2.resize(img2, size) 
            img1 = torch.from_numpy(img1).permute(2, 0, 1).float().unsqueeze(0)
            img2 = torch.from_numpy(img2).permute(2, 0, 1).float().unsqueeze(0)
            return img1, img2

        # index = index % len(self.image_list)
        # flow, valid = self.read_flow(index)
        # img1 = frame_utils.read_gen(self.image_list[index][0])
        # img2 = frame_utils.read_gen(self.image_list[index][1])
        # flow = np.array(flow).astype(np.float32)
        # img1 = np.array(img1).astype(np.uint8)
        # img2 = np.array(img2).astype(np.uint8)
        # # grayscale images
        # if len(img1.shape) == 2:
        #     img1 = np.tile(img1[...,None], (1, 1, 3))
        #     img2 = np.tile(img2[...,None], (1, 1, 3))
        # else:
        #     img1 = img1[..., :3]
        #     img2 = img2[..., :3]

        # if self.augmentor is not None:
        #     img1, img2, flow, valid = self.augmentor(img1, img2, flow, valid)

        # img1 = torch.from_numpy(img1).permute(2, 0, 1).float()
        # img2 = torch.from_numpy(img2).permute(2, 0, 1).float()
        # flow = torch.from_numpy(flow).permute(2, 0, 1).float()
        # valid = torch.from_numpy(valid)
        # valid = (valid >= 0.5) & ((~torch.isnan(flow)).all(dim=0)) & ((~torch.isinf(flow)).all(dim=0))
        # flow[torch.isinf(flow)] = 0
        # flow[torch.isnan(flow)] = 0
        # return img1, img2, flow, valid.float()

image1_file = 'assets/frame_0016.png'
image2_file = 'assets/frame_0018.png'
image1, image2 = preprocessing(image1_file, image2_file)

In [4]:
image1.shape

torch.Size([1, 3, 448, 448])

In [5]:
# from utils.utils import coords_grid, Padder, bilinear_sampler
import torch.nn.functional as F
class Padder:
    """ Pads images such that dimensions are divisible by factor """
    def __init__(self, dims, mode='sintel', factor=32):
        self.ht, self.wd = dims[-2:]
        if self.ht % factor == 0:
            pad_ht = 0
        else:
            pad_ht = (((self.ht + 8) // factor) + 1) * factor - self.ht
        if self.wd % factor == 0:
            pad_wd = 0
        else:
            pad_wd = (((self.wd + 8) // factor) + 1) * factor - self.wd
        if mode == 'sintel':
            self._pad = [pad_wd//2, pad_wd - pad_wd//2, pad_ht//2, pad_ht - pad_ht//2]
        else:
            self._pad = [pad_wd//2, pad_wd - pad_wd//2, 0, pad_ht]

    def pad(self, x):
        return F.pad(x, self._pad, mode='constant', value=0)

    def unpad(self, x):
        ht, wd = x.shape[-2:]
        c = [self._pad[2], ht-self._pad[3], self._pad[0], wd-self._pad[1]]
        return x[..., c[0]:c[1], c[2]:c[3]]
    
image1 = model.normalize_image(image1)
image2 = model.normalize_image(image2)
padder = Padder(image1.shape, factor=model.factor)
# image1 = padder.pad(image1)
# image2 = padder.pad(image2)

In [6]:
padder.ht, padder.wd, padder._pad

(448, 448, [0, 0, 0, 0])

In [7]:
image1 = padder.pad(image1)
image2 = padder.pad(image2)
image1.shape

torch.Size([1, 3, 448, 448])

In [8]:
image1.shape

torch.Size([1, 3, 448, 448])

In [9]:
out = model.export(image1.cuda(), image2.cuda())

fmap1_pretrain torch.Size([1, 64, 224, 224])
fnet torch.Size([1, 64, 224, 224])
fmap1_2x torch.Size([1, 64, 224, 224])
net torch.Size([1, 64, 224, 224])
refine_inp torch.Size([1, 64, 224, 224])
interpolate_pos_encoding torch.Size([1, 784, 384])
refine_outs torch.Size([1, 32, 224, 224])
net torch.Size([1, 64, 224, 224])
flow_update torch.Size([1, 6, 224, 224])
refine_inp torch.Size([1, 64, 224, 224])
interpolate_pos_encoding torch.Size([1, 784, 384])
refine_outs torch.Size([1, 32, 224, 224])
net torch.Size([1, 64, 224, 224])
flow_update torch.Size([1, 6, 224, 224])
refine_inp torch.Size([1, 64, 224, 224])
interpolate_pos_encoding torch.Size([1, 784, 384])
refine_outs torch.Size([1, 32, 224, 224])
net torch.Size([1, 64, 224, 224])
flow_update torch.Size([1, 6, 224, 224])
refine_inp torch.Size([1, 64, 224, 224])
interpolate_pos_encoding torch.Size([1, 784, 384])
refine_outs torch.Size([1, 32, 224, 224])
net torch.Size([1, 64, 224, 224])
flow_update torch.Size([1, 6, 224, 224])
refine_inp 

/home/rwang/py312/lib/python3.12/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
out['flow'][-1].shape, out['flow'][-2].shape
flow= out['flow'][-1]

(torch.Size([1, 2, 448, 448]), torch.Size([1, 2, 448, 448]))

In [15]:
flow= out['flow'][-1].cpu()
flow_vis = flow_to_image(flow[0].permute(1, 2, 0).cpu().detach().numpy(), convert_to_bgr=True)
cv2.imwrite(f"./flow_448.jpg", flow_vis)

True

In [1]:
from model.raft.raft import RAFT

from types import SimpleNamespace


default_conf = {
    "cfg": 'config/raft/eval/sintel-S.json',
    "url": None,
    'path': None,
    "device": 'cpu',
    }
conf = {}
args = SimpleNamespace(**{**default_conf, **conf})

if args.path is not None:
        model = RAFT(args)
        # load_ckpt(model, args.path)



xFormers not available
xFormers not available


ModuleNotFoundError: No module named 'update'

In [ ]:
from model.raft.raft import RAFT

from types import SimpleNamespace


default_conf = {
    "cfg": 'config/raft/eval/sintel-S.json',
    "url": None,
    'path': None,
    "device": 'cpu',
    }
conf = {}
args = SimpleNamespace(**{**default_conf, **conf})

if args.path is not None:
        model = RAFT(args)
        # load_ckpt(model, args.path)



xFormers not available
xFormers not available


ModuleNotFoundError: No module named 'update'

In [10]:
import torch
import cv2
import os

@torch.no_grad()
def demo_data(model, image1, image2, valid=None, tiling=False):
    H, W = image1.shape[2:]
    output = model.calc_flow(image1, image2)
    for i in range(len(output['flow'])):
        flow= output['flow'][i]
        flow_vis = flow_to_image(flow[0].permute(1, 2, 0).cpu().numpy(), convert_to_bgr=True)
        cv2.imwrite(f"./flow_{i}.jpg", flow_vis)

# wrapped_model = wrapped_model.cuda()
demo_data(wrapped_model, image1.unsqueeze(0).cuda(), image2.unsqueeze(0).cuda())

/home/rwang/py312/lib/python3.12/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
